# ORC Basics — 06: Hive Compatibility

ORC was born in the Hive ecosystem. Reading Hive-partitioned ORC tables,
handling Hive-style ACID, and working with Hive metastore metadata.

Topics: Hive-style partitioning, ACID ORC format, SerDe properties, metastore.


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/orc_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('orc-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

## Step 1 — Hive-Style Partitioned ORC

Hive stores partition values in directory names: dt=2024-01/

In [ ]:

HIVE_PATH = f"{DATA_DIR}/hive_partitioned"
import random, datetime, pathlib
random.seed(42)

# Write Hive-style partitioned ORC
import os; CATS=["Electronics","Clothing","Books"]
for cat in CATS:
    for month in [1,2,3]:
        part_path = f"{HIVE_PATH}/category={cat}/month={month}"
        pathlib.Path(part_path).mkdir(parents=True, exist_ok=True)
        rows = [(i+month*1000, random.randint(1,1000), cat, f"US", round(random.uniform(10,500),2),
                 datetime.date(2024,month,random.randint(1,28)))
                for i in range(1000)]
        df_part = spark.createDataFrame(rows, ["order_id","customer_id","category","country","revenue","order_date"])
        df_part.coalesce(1).write.mode("overwrite").orc(part_path)

print("Hive-style partitioned ORC:")
for cat in CATS:
    for month in [1,2]:
        path = f"{HIVE_PATH}/category={cat}/month={month}"
        cnt = spark.read.orc(path).count()
        print(f"  category={cat}/month={month}: {cnt} rows")


## Step 2 — Reading Hive-Partitioned ORC

Spark automatically discovers partitions.

In [ ]:

# Spark reads all partitions + infers partition columns from directory names
df_hive = spark.read.orc(HIVE_PATH)
print("Schema (includes partition columns from directory names):")
df_hive.printSchema()
print(f"Total rows: {df_hive.count():,}")
print()
# Partition pruning works automatically
df_elec = df_hive.filter(F.col("category")=="Electronics")
print(f"Electronics only (partition pruning): {df_elec.count():,} rows")
df_elec.explain(mode="simple")


## Step 3 — Handling Hive ACID ORC

What you'll see and how to read it.

In [ ]:

print("""
Hive ACID ORC directory structure you may encounter:
  warehouse/table/
    base_0000001/
      bucket_00000.orc   ← full data snapshot
    delta_0000002_0000002/
      bucket_00000.orc   ← INSERT delta files
    delete_delta_0000003_0000003/
      bucket_00000.orc   ← DELETE marker files

Reading Hive ACID in Spark:
  spark.read.orc("/hive/warehouse/table/")

Spark 3.0+ reads ACID ORC correctly by default.
It merges base + delta files transparently.

If you get errors reading ACID ORC:
  1. Ensure spark-hive JAR is on classpath
  2. Set spark.sql.hive.convertInsertingPartitionedTable = false
  3. Or use Hive JDBC/ODBC to export to plain ORC first
""")


## Step 4 — ORC SerDe Properties

Properties you may see in Hive DDL.

In [ ]:

print("""
Common ORC SerDe properties in Hive DDL:
  STORED AS ORC
  TBLPROPERTIES (
    'orc.compress'                = 'SNAPPY',
    'orc.compress.size'           = '262144',   -- 256 KB compression buffer
    'orc.stripe.size'             = '67108864', -- 64 MB
    'orc.row.index.stride'        = '10000',
    'orc.bloom.filter.columns'    = 'col1,col2',
    'orc.bloom.filter.fpp'        = '0.05',
    'transactional'               = 'true'      -- enables ACID
  )

Reading in Spark — these properties are embedded in the ORC file footer.
Spark automatically respects the compression and stripe settings.
You don't need to specify them in spark.read.orc() — they're auto-detected.
""")


## Summary



In [ ]:

print("""
ORC + Hive compatibility:
  Reading Hive-partitioned ORC: spark.read.orc('/hive/warehouse/table/')
  Spark auto-discovers: partition columns, stripe size, compression, bloom filters
  
  Hive ACID: Spark 3.0+ reads transparently (base + delta merge)
  
  Writing for Hive compatibility:
    .option('orc.stripe.size', '67108864')      -- match Hive default
    .option('orc.row.index.stride', '10000')    -- Hive default
    .option('compression', 'zlib')              -- Hive default compression
""")
